# Phase 2 — DIV2K Data Preparation
**Goal:** download HR images, save fixed image-level splits, inspect aligned LR/HR pairs, and check a batch.

Use a Colab GPU runtime. This notebook can run independently after phase 1.

**Download:** the full official training and validation HR archives are several GB, even in the 100-image smoke mode. Budget at least 12 GB free on Colab and 5 GB on Drive for cached archives. First download may take 5–20 minutes or longer depending on the server and connection.

Source: [official DIV2K page](https://data.vision.ee.ethz.ch/cvl/DIV2K/). Images are for academic research; image copyright remains with the original owners. They are not covered by this repository's code license.


## 1. Load the current repository
An existing clean checkout is updated with a fast-forward pull. If local tracked edits exist, this cell stops so you can save them first. It never resets your edits.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/SRGAN-Image-Super-Resolution")
REPO_URL = "https://github.com/ennaguiyoussef/SRGAN-Image-Super-Resolution.git"
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    if not (PROJECT_ROOT / ".git").is_dir():
        raise RuntimeError("Project folder exists but is not a Git checkout.")
    changes = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    )
    if changes.strip():
        raise RuntimeError("Save local tracked edits before updating this checkout.")
    branch = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "main":
        raise RuntimeError("Select the main branch before running this notebook.")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project checkout ready.")


## 2. Connect Drive and choose smoke mode
`SMOKE_MODE = True` selects a fixed 100-image subset from the 800 training images. Set it to `False` later for all 800. Validation/test membership never changes with this switch.


In [ ]:
from google.colab import drive
from src.setup import load_config, prepare_directories, seed_everything

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/SRGAN-Image-Super-Resolution")
paths = prepare_directories(PROJECT_ROOT, OUTPUT_ROOT)
CACHE_ROOT = OUTPUT_ROOT / "dataset_cache"
DATA_ROOT = paths["data"] / "DIV2K"
config = load_config(PROJECT_ROOT / "configs/srgan.yaml")
seed_everything(config["seed"])
SMOKE_MODE = True
print("Mode:", "100 training images" if SMOKE_MODE else "800 training images")


## 3. Download and extract DIV2K
Completed archives are reused from Drive after a Colab reset. Extracted images stay in Colab storage for faster reads. Interrupted downloads restart when rerun; keep this cell running until both archives finish.


In [ ]:
import shutil
from src.data import download_div2k

for label, location, minimum_gb in [
    ("Colab", Path("/content"), 12),
    ("Drive cache", OUTPUT_ROOT, 5),
]:
    free_gb = shutil.disk_usage(location).free / (1024 ** 3)
    print(f"{label}: {free_gb:.1f} GB reported free")
    if free_gb < minimum_gb:
        print(f"Check storage before continuing: budget approximately {minimum_gb} GB.")
print("Drive quota should also be checked in Google Drive; mounted filesystem reports may differ.")
download_div2k(DATA_ROOT, CACHE_ROOT)


## 4. Save and validate image-level splits
- Training: official HR images 0001–0800; smoke mode uses a fixed subset of 100.
- Validation: 50 images from official validation HR.
- Held-out test: the other 50; do not use them to tune the model.

This is our custom split, not the official DIV2K challenge test set. Splitting happens before creating crops. The manifest is saved on Drive and refuses silent changes.


In [ ]:
from src.data import save_manifest, split_manifest

manifest = split_manifest(seed=config["seed"])
MANIFEST_PATH = OUTPUT_ROOT / "splits" / "div2k_seed42.json"
save_manifest(MANIFEST_PATH, manifest)
groups = [set(manifest[k]) for k in ("train", "validation", "test")]
assert all(not groups[i] & groups[j] for i in range(3) for j in range(i))
for key in ("train", "validation", "test"):
    assert all((DATA_ROOT / name).is_file() for name in manifest[key])
    print(f"{key}: {len(manifest[key])} images")
train_names = manifest["train_smoke"] if SMOKE_MODE else manifest["train"]
print("Active training images:", len(train_names))
print("Saved split:", MANIFEST_PATH)


## 5. Build the datasets and inspect one pair
The HR crop is taken first; LR is derived from exactly that crop with **Pillow bicubic**. Pillow bicubic differs from the official MATLAB degradation; do not compare future metrics directly to the official leaderboard.

Tensor conventions:
- LR: RGB, float32, shape `(3, 24, 24)`, range **[0, 1]**.
- HR: RGB, float32, shape `(3, 96, 96)`, range **[-1, 1]**.

Training uses random crops. Validation uses repeatable center crops. Center-crop checks here are not the final full-image evaluation protocol.


In [ ]:
from src.dataset import SuperResolutionDataset

train_dataset = SuperResolutionDataset(
    DATA_ROOT, train_names, hr_size=config["hr_crop_size"],
    scale=config["scale_factor"], training=True,
)
validation_dataset = SuperResolutionDataset(
    DATA_ROOT, manifest["validation"], hr_size=config["hr_crop_size"],
    scale=config["scale_factor"], training=False,
)
lr, hr = validation_dataset[0]
assert tuple(lr.shape) == (3, 24, 24)
assert tuple(hr.shape) == (3, 96, 96)
assert 0 <= lr.min().item() <= lr.max().item() <= 1
assert -1 <= hr.min().item() <= hr.max().item() <= 1
print("LR shape:", tuple(lr.shape), "| range:", (lr.min().item(), lr.max().item()))
print("HR shape:", tuple(hr.shape), "| range:", (hr.min().item(), hr.max().item()))


## 6. Visualize LR, bicubic baseline, and HR
All panels have equal display size for comparison. The first panel uses nearest-neighbor display to expose the LR pixels. The middle panel is ordinary bicubic interpolation, not a model prediction. Test images remain untouched.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row in range(3):
    lr, hr = validation_dataset[row]
    lr_image = lr.permute(1, 2, 0).numpy()
    hr_image = ((hr + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    bicubic = Image.fromarray(np.rint(lr_image * 255).astype(np.uint8)).resize(
        (96, 96), Image.Resampling.BICUBIC
    )
    panels = [(lr_image, "LR 24 x 24"), (bicubic, "Bicubic 96 x 96"), (hr_image, "HR reference 96 x 96")]
    for col, (image, title) in enumerate(panels):
        axes[row, col].imshow(image, interpolation="nearest")
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
fig.tight_layout()
preview_path = paths["results"] / "phase2_lr_bicubic_hr.png"
fig.savefig(preview_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved preview:", preview_path)


## 7. Verify a batch and record data settings
This check uses zero worker subprocesses for easy diagnosis. Training worker settings will be tuned later. No model is trained here.


In [ ]:
import torch
from torch.utils.data import DataLoader

generator = torch.Generator().manual_seed(config["seed"])
train_loader = DataLoader(
    train_dataset, batch_size=config["batch_size"], shuffle=True,
    num_workers=0, generator=generator, pin_memory=torch.cuda.is_available(),
)
lr_batch, hr_batch = next(iter(train_loader))
assert tuple(lr_batch.shape[1:]) == (3, 24, 24)
assert tuple(hr_batch.shape[1:]) == (3, 96, 96)
assert torch.isfinite(lr_batch).all() and torch.isfinite(hr_batch).all()
print("LR batch:", tuple(lr_batch.shape))
print("HR batch:", tuple(hr_batch.shape))
settings = {
    "dataset": "DIV2K", "smoke_mode": SMOKE_MODE,
    "active_training_images": len(train_dataset),
    "validation_images": len(validation_dataset), "held_out_test_images": len(manifest["test"]),
    "manifest": str(MANIFEST_PATH), "degradation": "Pillow bicubic, HR crop then resize",
    "lr_range": [0, 1], "hr_range": [-1, 1],
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "config": config,
}
(paths["logs"] / "phase2_data_settings.json").write_text(
    json.dumps(settings, indent=2) + "\n", encoding="utf-8"
)
print("Phase 2 complete: splits, LR/HR pairs, visualization, and batch checks passed.")


## Next step
Inspect the saved comparison: which panel contains the original fine details, and why does enlarging the LR image with bicubic fail to recover all of them?

Next: Phase 3 — understand and implement the generator and discriminator.
